# 02 - Single-Qubit Gates

**Prerequisites:** Notebook 01. Familiarity with qubits, superposition, and the Bloch sphere.

Every quantum computation is built from **gates** -- unitary transformations that rotate the state of a qubit on the Bloch sphere. In this notebook we explore the standard single-qubit gates available in Goqu:

- **Pauli gates** (X, Y, Z) -- 180-degree rotations around the three Bloch sphere axes
- **Hadamard** (H) -- creates equal superposition
- **Phase gates** (S, T) -- fractional Z-rotations forming the S^2 = Z, T^2 = S, T^4 = Z chain
- **Rotation gates** (RX, RY, RZ) -- continuous rotations, universal for single-qubit operations
- **Gate inverses** and **gate powers**

Geometrically, every single-qubit gate is a rotation of the Bloch sphere. A gate's 2x2 unitary matrix encodes the axis and angle of that rotation. We will visualize these rotations using Goqu's Bloch sphere renderer.

By the end of this notebook you will be able to:

1. **Describe** the Pauli, phase, and rotation gates and their matrix representations.
2. **Implement** any single-qubit rotation using RX, RY, RZ gates.
3. **Verify** the S²=Z, T²=S, T⁴=Z hierarchy using gate power operations.
4. **Explain** why any single-qubit unitary can be decomposed into three rotations.

In [1]:
import (
	"fmt"
	"math"
	"math/cmplx"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/gate"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## Pauli Gates: X, Y, Z

The three Pauli gates are 180-degree rotations around the X, Y, and Z axes of the Bloch sphere:

| Gate | Action | Matrix |
|------|--------|--------|
| **X** (bit-flip) | \|0> -> \|1>, \|1> -> \|0> | `[[0,1],[1,0]]` |
| **Y** | \|0> -> i\|1>, \|1> -> -i\|0> | `[[0,-i],[i,0]]` |
| **Z** (phase-flip) | \|0> -> \|0>, \|1> -> -\|1> | `[[1,0],[0,-1]]` |

All three are **self-inverse**: applying the same gate twice returns the qubit to its original state.

In [2]:
%%
// Apply X gate to |0> -- should produce |1>
c, _ := builder.New("X-gate", 1).X(0).Build()
sim := statevector.New(1)
sim.Evolve(c)
sv := sim.StateVector()
fmt.Println("X|0> =", sv)
fmt.Println("Circuit:")
gonbui.DisplayHTML(draw.SVG(c))

// Print the Pauli gate matrices
for _, info := range []struct {
	name string
	g    gate.Gate
}{
	{"X", gate.X},
	{"Y", gate.Y},
	{"Z", gate.Z},
} {
	m := info.g.Matrix()
	fmt.Printf("%s matrix: [[%v, %v], [%v, %v]]\n", info.name, m[0], m[1], m[2], m[3])
}

X|0> = [(0+0i) (1+0i)]
Circuit:
X matrix: [[(0+0i), (1+0i)], [(1+0i), (0+0i)]]
Y matrix: [[(0+0i), (0-1i)], [(0+1i), (0+0i)]]
Z matrix: [[(1+0i), (0+0i)], [(0+0i), (-1+0i)]]


q0 
 
 X

## The Hadamard Gate

The Hadamard gate **H** is arguably the most important single-qubit gate. It maps computational basis states to superposition states:

- H|0> = |+> = (|0> + |1>) / sqrt(2)
- H|1> = |-> = (|0> - |1>) / sqrt(2)

On the Bloch sphere, H is a 180-degree rotation around the axis halfway between X and Z. Like the Pauli gates, H is self-inverse: H^2 = I.

In [3]:
%%
// Apply H to |0> and visualize on the Bloch sphere
c, _ := builder.New("H-gate", 1).H(0).Build()
sim := statevector.New(1)
sim.Evolve(c)
sv := sim.StateVector()
fmt.Println("H|0> =", sv)
fmt.Printf("  |0> amplitude: %.4f\n", sv[0])
fmt.Printf("  |1> amplitude: %.4f\n", sv[1])
fmt.Printf("  Both equal 1/sqrt(2) = %.4f\n", 1/math.Sqrt(2))
fmt.Println()
fmt.Println("Circuit:")
gonbui.DisplayHTML(draw.SVG(c))

// Render on the Bloch sphere -- |+> points along the +X axis
gonbui.DisplayHTML(viz.Bloch(sv, viz.WithTitle("|+> = H|0>")))

H|0> = [(0.7071067811865476+0i) (0.7071067811865476+0i)]
  |0> amplitude: (0.7071+0.0000i)
  |1> amplitude: (0.7071+0.0000i)
  Both equal 1/sqrt(2) = 0.7071

Circuit:


q0 
 
 H

|+> = H|0> 
<path d="M438.6,230.6 L445.0,225.9 L450.4,220.9 L454.5,215.8 L457.6,210.6 L459.4,205.3 L460.0,200.0 L459.4,194.7 L457.6,189.4 L454.5,184.2 L450.4,179.1 L445.0,174.1 L438.6,169.4 L431.1,164.9 L422.6,160.6 L413.1,156.7 L402.8,153.1 L391.8,149.8 L380.0,147.0 L367.6,144.5 L354.7,142.5 L341.4,140.9 L327.8,139.7 L313.9,139.0 L300.0,138.8 L286.1,139.0 L272.2,139.7 L258.6,140.9 L245.3,142.5 L232.4,144.5 L220.0,147.0 L208.2,149.8 L197.2,153.1 L186.9,156.7 L177.4,160.6 L168.9,164.9 L161.4,169.4 L155.0,174.1 L149.6,179.1 L145.5,184.2 L142.4,189.4 L140.6,194.7 L140.0,200.0 L140.6,205.3 L142.4,210.6 L145.5,215.8 L149.6,220.9 L155.0,225.9 L161.4,230.6 L168.9,235.1 L177.4,239.4 L186.9,243.3 L197.2,246.9 L208.2,250.2 L220.0,253.0 L232.4,255.5 L245.3,257.5 L258.6,259.1 L272.2,260.3 L286.1,261.0 L300.0,261.2 L313.9,261.0 L327.8,260.3 L341.4,259.1 L354.7,257.5 L367.6,255.5 L380.0,253.0 L391.8,250.2 L402.8,246.9 L413.1,243.3 L422.6,239.4 L431.1,235.1 L438.6,230.6 Z" fill="none" stroke="#AAAAAA" stroke-width="0.5" opacity="0.4"/>
<path d="M438.6,230.6 L438.0,217.6 L436.5,204.5 L433.8,191.3 L430.2,178.2 L425.6,165.3 L420.0,152.6 L413.5,140.3 L406.1,128.4 L398.0,117.1 L389.1,106.4 L379.5,96.5 L369.3,87.3 L358.6,79.0 L347.4,71.6 L335.9,65.1 L324.1,59.7 L312.1,55.4 L300.0,52.2 L287.9,50.1 L275.9,49.1 L264.1,49.3 L252.6,50.6 L241.4,53.1 L230.7,56.7 L220.5,61.4 L210.9,67.1 L202.0,73.8 L193.9,81.5 L186.5,90.1 L180.0,99.6 L174.4,109.8 L169.8,120.7 L166.2,132.2 L163.5,144.2 L162.0,156.6 L161.4,169.4 L162.0,182.4 L163.5,195.5 L166.2,208.7 L169.8,221.8 L174.4,234.7 L180.0,247.4 L186.5,259.7 L193.9,271.6 L202.0,282.9 L210.9,293.6 L220.5,303.5 L230.7,312.7 L241.4,321.0 L252.6,328.4 L264.1,334.9 L275.9,340.3 L287.9,344.6 L300.0,347.8 L312.1,349.9 L324.1,350.9 L335.9,350.7 L347.4,349.4 L358.6,346.9 L369.3,343.3 L379.5,338.6 L389.1,332.9 L398.0,326.2 L406.1,318.5 L413.5,309.9 L420.0,300.4 L425.6,290.2 L430.2,279.3 L433.8,267.8 L436.5,255.8 L438.0,243.4 L438.6,230.6 Z" fill="none" stroke="#AAAAAA" stroke-width="0.5" opacity="0.4"/>
<path d="M380.0,147.0 L379.7,134.3 L378.8,122.1 L377.3,110.5 L375.2,99.6 L372.5,89.5 L369.3,80.2 L365.5,71.8 L361.3,64.4 L356.6,58.0 L351.4,52.7 L345.9,48.5 L340.0,45.5 L333.8,43.6 L327.4,43.0 L320.7,43.5 L313.9,45.2 L307.0,48.1 L300.0,52.2 L293.0,57.4 L286.1,63.6 L279.3,70.9 L272.6,79.2 L266.2,88.4 L260.0,98.5 L254.1,109.3 L248.6,120.8 L243.4,133.0 L238.7,145.6 L234.5,158.7 L230.7,172.0 L227.5,185.6 L224.8,199.3 L222.7,213.0 L221.2,226.6 L220.3,239.9 L220.0,253.0 L220.3,265.7 L221.2,277.9 L222.7,289.5 L224.8,300.4 L227.5,310.5 L230.7,319.8 L234.5,328.2 L238.7,335.6 L243.4,342.0 L248.6,347.3 L254.1,351.5 L260.0,354.5 L266.2,356.4 L272.6,357.0 L279.3,356.5 L286.1,354.8 L293.0,351.9 L300.0,347.8 L307.0,342.6 L313.9,336.4 L320.7,329.1 L327.4,320.8 L333.8,311.6 L340.0,301.5 L345.9,290.7 L351.4,279.2 L356.6,267.0 L361.3,254.4 L365.5,241.3 L369.3,228.0 L372.5,214.4 L375.2,200.7 L377.3,187.0 L378.8,173.4 L379.7,160.1 L380.0,147.0 Z" fill="none" stroke="#AAAAAA" stroke-width="0.5" opacity="0.4"/>
 
 
 
 
 |+⟩ 
 |-⟩ 
 |+i⟩ 
 |-i⟩ 
 |0⟩ 
 |1⟩

## Phase Gates: S and T

Phase gates rotate the qubit around the Z axis by fractional angles:

| Gate | Phase on |1> | Relation |
|------|------------|----------|
| **Z** | e^(i*pi) = -1 | pi rotation |
| **S** | e^(i*pi/2) = i | S^2 = Z |
| **T** | e^(i*pi/4) | T^2 = S, T^4 = Z |

The T gate is critical for universal quantum computing: the **Clifford+T** gate set can approximate any unitary to arbitrary precision (Solovay-Kitaev theorem).

Let's verify the S^2 = Z and T^2 = S chain by comparing gate matrices.

In [4]:
%%
// Helper to compare two gate matrices
matricesEqual := func(a, b []complex128, tol float64) bool {
	if len(a) != len(b) {
		return false
	}
	for i := range a {
		if cmplx.Abs(a[i]-b[i]) > tol {
			return false
		}
	}
	return true
}

// S^2 = Z
s2 := gate.Pow(gate.S, 2)
fmt.Println("S^2 matrix:", s2.Matrix())
fmt.Println("Z   matrix:", gate.Z.Matrix())
fmt.Println("S^2 == Z?", matricesEqual(s2.Matrix(), gate.Z.Matrix(), 1e-10))
fmt.Println()

// T^2 = S
t2 := gate.Pow(gate.T, 2)
fmt.Println("T^2 matrix:", t2.Matrix())
fmt.Println("S   matrix:", gate.S.Matrix())
fmt.Println("T^2 == S?", matricesEqual(t2.Matrix(), gate.S.Matrix(), 1e-10))
fmt.Println()

// T^4 = Z
t4 := gate.Pow(gate.T, 4)
fmt.Println("T^4 matrix:", t4.Matrix())
fmt.Println("Z   matrix:", gate.Z.Matrix())
fmt.Println("T^4 == Z?", matricesEqual(t4.Matrix(), gate.Z.Matrix(), 1e-10))

S^2 matrix: [(1+0i) (0+0i) (0+0i) (-1+0i)]
Z   matrix: [(1+0i) (0+0i) (0+0i) (-1+0i)]
S^2 == Z? true

T^2 matrix: [(1+0i) (0+0i) (0+0i) (4.266421588589642e-17+1.0000000000000002i)]
S   matrix: [(1+0i) (0+0i) (0+0i) (0+1i)]
T^2 == S? true

T^4 matrix: [(1+0i) (0+0i) (0+0i) (-1.0000000000000004+1.143450299824811e-16i)]
Z   matrix: [(1+0i) (0+0i) (0+0i) (-1+0i)]
T^4 == Z? true


## Rotation Gates: RX, RY, RZ

Rotation gates provide **continuous** rotation around the Bloch sphere axes:

- **RX(theta)** = exp(-i * theta/2 * X) -- rotation around the X axis
- **RY(theta)** = exp(-i * theta/2 * Y) -- rotation around the Y axis  
- **RZ(theta)** = exp(-i * theta/2 * Z) -- rotation around the Z axis

These gates are **universal**: any single-qubit unitary can be decomposed as a sequence of rotations RZ(a) * RY(b) * RZ(c) (the ZYZ Euler decomposition).

Let's sweep RX from 0 to 2*pi and observe how the statevector evolves.

In [5]:
%%
// RX(theta) sweep from 0 to 2*pi
fmt.Println("RX(theta)|0> at several angles:")
fmt.Println("------------------------------------------")
angles := []struct {
	name  string
	theta float64
}{
	{"0", 0},
	{"pi/4", math.Pi / 4},
	{"pi/2", math.Pi / 2},
	{"pi", math.Pi},
	{"3pi/2", 3 * math.Pi / 2},
	{"2pi", 2 * math.Pi},
}

for _, a := range angles {
	c, _ := builder.New("rx", 1).RX(a.theta, 0).Build()
	sim := statevector.New(1)
	sim.Evolve(c)
	sv := sim.StateVector()
	p0 := real(sv[0])*real(sv[0]) + imag(sv[0])*imag(sv[0])
	p1 := real(sv[1])*real(sv[1]) + imag(sv[1])*imag(sv[1])
	fmt.Printf("  RX(%6s)|0> = [%8.4f, %8.4f]  P(|0>)=%.3f  P(|1>)=%.3f\n",
		a.name, sv[0], sv[1], p0, p1)
}

// Visualize RX(pi/2)|0> on the Bloch sphere
c, _ := builder.New("rx-pi/2", 1).RX(math.Pi/2, 0).Build()
sim := statevector.New(1)
sim.Evolve(c)
gonbui.DisplayHTML(viz.Bloch(sim.StateVector(), viz.WithTitle("RX(pi/2)|0>")))

RX(theta)|0> at several angles:
------------------------------------------
  RX(     0)|0> = [(  1.0000 +0.0000i), (  0.0000 +0.0000i)]  P(|0>)=1.000  P(|1>)=0.000
  RX(  pi/4)|0> = [(  0.9239 +0.0000i), (  0.0000 -0.3827i)]  P(|0>)=0.854  P(|1>)=0.146
  RX(  pi/2)|0> = [(  0.7071 +0.0000i), (  0.0000 -0.7071i)]  P(|0>)=0.500  P(|1>)=0.500
  RX(    pi)|0> = [(  0.0000 +0.0000i), (  0.0000 -1.0000i)]  P(|0>)=0.000  P(|1>)=1.000
  RX( 3pi/2)|0> = [( -0.7071 +0.0000i), (  0.0000 -0.7071i)]  P(|0>)=0.500  P(|1>)=0.500
  RX(   2pi)|0> = [( -1.0000 +0.0000i), (  0.0000 -0.0000i)]  P(|0>)=1.000  P(|1>)=0.000


RX(pi/2)|0> 
<path d="M438.6,230.6 L445.0,225.9 L450.4,220.9 L454.5,215.8 L457.6,210.6 L459.4,205.3 L460.0,200.0 L459.4,194.7 L457.6,189.4 L454.5,184.2 L450.4,179.1 L445.0,174.1 L438.6,169.4 L431.1,164.9 L422.6,160.6 L413.1,156.7 L402.8,153.1 L391.8,149.8 L380.0,147.0 L367.6,144.5 L354.7,142.5 L341.4,140.9 L327.8,139.7 L313.9,139.0 L300.0,138.8 L286.1,139.0 L272.2,139.7 L258.6,140.9 L245.3,142.5 L232.4,144.5 L220.0,147.0 L208.2,149.8 L197.2,153.1 L186.9,156.7 L177.4,160.6 L168.9,164.9 L161.4,169.4 L155.0,174.1 L149.6,179.1 L145.5,184.2 L142.4,189.4 L140.6,194.7 L140.0,200.0 L140.6,205.3 L142.4,210.6 L145.5,215.8 L149.6,220.9 L155.0,225.9 L161.4,230.6 L168.9,235.1 L177.4,239.4 L186.9,243.3 L197.2,246.9 L208.2,250.2 L220.0,253.0 L232.4,255.5 L245.3,257.5 L258.6,259.1 L272.2,260.3 L286.1,261.0 L300.0,261.2 L313.9,261.0 L327.8,260.3 L341.4,259.1 L354.7,257.5 L367.6,255.5 L380.0,253.0 L391.8,250.2 L402.8,246.9 L413.1,243.3 L422.6,239.4 L431.1,235.1 L438.6,230.6 Z" fill="none" stroke="#AAAAAA" stroke-width="0.5" opacity="0.4"/>
<path d="M438.6,230.6 L438.0,217.6 L436.5,204.5 L433.8,191.3 L430.2,178.2 L425.6,165.3 L420.0,152.6 L413.5,140.3 L406.1,128.4 L398.0,117.1 L389.1,106.4 L379.5,96.5 L369.3,87.3 L358.6,79.0 L347.4,71.6 L335.9,65.1 L324.1,59.7 L312.1,55.4 L300.0,52.2 L287.9,50.1 L275.9,49.1 L264.1,49.3 L252.6,50.6 L241.4,53.1 L230.7,56.7 L220.5,61.4 L210.9,67.1 L202.0,73.8 L193.9,81.5 L186.5,90.1 L180.0,99.6 L174.4,109.8 L169.8,120.7 L166.2,132.2 L163.5,144.2 L162.0,156.6 L161.4,169.4 L162.0,182.4 L163.5,195.5 L166.2,208.7 L169.8,221.8 L174.4,234.7 L180.0,247.4 L186.5,259.7 L193.9,271.6 L202.0,282.9 L210.9,293.6 L220.5,303.5 L230.7,312.7 L241.4,321.0 L252.6,328.4 L264.1,334.9 L275.9,340.3 L287.9,344.6 L300.0,347.8 L312.1,349.9 L324.1,350.9 L335.9,350.7 L347.4,349.4 L358.6,346.9 L369.3,343.3 L379.5,338.6 L389.1,332.9 L398.0,326.2 L406.1,318.5 L413.5,309.9 L420.0,300.4 L425.6,290.2 L430.2,279.3 L433.8,267.8 L436.5,255.8 L438.0,243.4 L438.6,230.6 Z" fill="none" stroke="#AAAAAA" stroke-width="0.5" opacity="0.4"/>
<path d="M380.0,147.0 L379.7,134.3 L378.8,122.1 L377.3,110.5 L375.2,99.6 L372.5,89.5 L369.3,80.2 L365.5,71.8 L361.3,64.4 L356.6,58.0 L351.4,52.7 L345.9,48.5 L340.0,45.5 L333.8,43.6 L327.4,43.0 L320.7,43.5 L313.9,45.2 L307.0,48.1 L300.0,52.2 L293.0,57.4 L286.1,63.6 L279.3,70.9 L272.6,79.2 L266.2,88.4 L260.0,98.5 L254.1,109.3 L248.6,120.8 L243.4,133.0 L238.7,145.6 L234.5,158.7 L230.7,172.0 L227.5,185.6 L224.8,199.3 L222.7,213.0 L221.2,226.6 L220.3,239.9 L220.0,253.0 L220.3,265.7 L221.2,277.9 L222.7,289.5 L224.8,300.4 L227.5,310.5 L230.7,319.8 L234.5,328.2 L238.7,335.6 L243.4,342.0 L248.6,347.3 L254.1,351.5 L260.0,354.5 L266.2,356.4 L272.6,357.0 L279.3,356.5 L286.1,354.8 L293.0,351.9 L300.0,347.8 L307.0,342.6 L313.9,336.4 L320.7,329.1 L327.4,320.8 L333.8,311.6 L340.0,301.5 L345.9,290.7 L351.4,279.2 L356.6,267.0 L361.3,254.4 L365.5,241.3 L369.3,228.0 L372.5,214.4 L375.2,200.7 L377.3,187.0 L378.8,173.4 L379.7,160.1 L380.0,147.0 Z" fill="none" stroke="#AAAAAA" stroke-width="0.5" opacity="0.4"/>
 
 
 
 
 |+⟩ 
 |-⟩ 
 |+i⟩ 
 |-i⟩ 
 |0⟩ 
 |1⟩

## Gate Inverses

Every quantum gate is unitary, which means it has an inverse (its conjugate transpose, or **adjoint**). Applying a gate followed by its inverse returns the qubit to its original state.

In Goqu, call `.Inverse()` on any gate to get its adjoint:
- `gate.H.Inverse()` returns H (since H is self-inverse)
- `gate.S.Inverse()` returns S-dagger (Sdg)
- `gate.T.Inverse()` returns T-dagger (Tdg)

In [6]:
%%
// Apply a gate then its inverse -- should return to |0>
fmt.Println("Gate -> Inverse -> back to |0>:")
fmt.Println("-------------------------------")

for _, info := range []struct {
	name string
	g    gate.Gate
}{
	{"H", gate.H},
	{"S", gate.S},
	{"T", gate.T},
	{"X", gate.X},
	{"RX(pi/3)", gate.RX(math.Pi / 3)},
} {
	inv := info.g.Inverse()
	c, _ := builder.New("inv", 1).
		Apply(info.g, 0).
		Apply(inv, 0).
		Build()
	sim := statevector.New(1)
	sim.Evolve(c)
	sv := sim.StateVector()
	p0 := real(sv[0])*real(sv[0]) + imag(sv[0])*imag(sv[0])
	fmt.Printf("  %10s then %s^dag : P(|0>) = %.6f\n", info.name, info.name, p0)
}

// Show that S.Inverse() == Sdg
fmt.Println()
fmt.Println("S.Inverse() name:", gate.S.Inverse().Name())
fmt.Println("T.Inverse() name:", gate.T.Inverse().Name())

Gate -> Inverse -> back to |0>:
-------------------------------
           H then H^dag : P(|0>) = 1.000000
           S then S^dag : P(|0>) = 1.000000
           T then T^dag : P(|0>) = 1.000000
           X then X^dag : P(|0>) = 1.000000
    RX(pi/3) then RX(pi/3)^dag : P(|0>) = 1.000000

S.Inverse() name: S†
T.Inverse() name: T†


## Global Phase

Two quantum states that differ only by a **global phase** (a multiplicative factor $e^{i\phi}$) are physically indistinguishable -- they produce identical measurement probabilities and identical expectation values. For example, $|0\rangle$ and $e^{i\pi/4}|0\rangle$ are the same physical state.

This means gate decompositions may differ by a global phase and still be physically equivalent. When we say "RY(pi/2) followed by RZ(pi) is equivalent to H," we mean they produce the same measurement statistics, even though their matrices may differ by a factor of $e^{i\phi}$.

Global phase is different from **relative phase** (the phase difference between amplitudes), which IS physically observable through interference.

## Predict, Then Verify

**Question:** What state does $\text{RY}(\pi/2)|0\rangle$ produce? Where does it sit on the Bloch sphere?

**Pause and predict** before running the next cell.

*Hint: Think about what a 90-degree rotation around the Y-axis does to the north pole of the Bloch sphere.*

In [7]:
%%
// RY(pi/2)|0> should produce |+> (same as H|0>)
cRY, _ := builder.New("ry", 1).RY(math.Pi/2, 0).Build()
cH, _ := builder.New("h", 1).H(0).Build()

simRY := statevector.New(1)
simRY.Evolve(cRY)
svRY := simRY.StateVector()

simH := statevector.New(1)
simH.Evolve(cH)
svH := simH.StateVector()

fmt.Println("RY(pi/2)|0> =", svRY)
fmt.Println("H|0>        =", svH)

// They both have equal |0> and |1> amplitudes (both real, both 1/sqrt(2))
fmt.Printf("\nBoth produce equal superposition with P(|0>) = P(|1>) = 0.5\n")
fmt.Printf("RY version: P(|0>)=%.4f, P(|1>)=%.4f\n",
	real(svRY[0])*real(svRY[0])+imag(svRY[0])*imag(svRY[0]),
	real(svRY[1])*real(svRY[1])+imag(svRY[1])*imag(svRY[1]))
fmt.Printf("H  version: P(|0>)=%.4f, P(|1>)=%.4f\n",
	real(svH[0])*real(svH[0])+imag(svH[0])*imag(svH[0]),
	real(svH[1])*real(svH[1])+imag(svH[1])*imag(svH[1]))

// Visualize both side by side
gonbui.DisplayHTML(
	"<div style='display:flex;gap:20px'>" +
		viz.Bloch(svRY, viz.WithTitle("RY(pi/2)|0>"), viz.WithSize(300, 300)) +
		viz.Bloch(svH, viz.WithTitle("H|0>"), viz.WithSize(300, 300)) +
		"</div>")

RY(pi/2)|0> = [(0.7071067811865476+0i) (0.7071067811865475+0i)]
H|0>        = [(0.7071067811865476+0i) (0.7071067811865476+0i)]

Both produce equal superposition with P(|0>) = P(|1>) = 0.5
RY version: P(|0>)=0.5000, P(|1>)=0.5000
H  version: P(|0>)=0.5000, P(|1>)=0.5000


RY(pi/2)|0> 
<path d="M245.3,171.0 L249.7,167.8 L253.4,164.4 L256.3,160.9 L258.3,157.3 L259.6,153.7 L260.0,150.0 L259.6,146.3 L258.3,142.7 L256.3,139.1 L253.4,135.6 L249.7,132.2 L245.3,129.0 L240.1,125.9 L234.3,122.9 L227.8,120.2 L220.7,117.8 L213.1,115.5 L205.0,113.5 L196.5,111.8 L187.6,110.4 L178.5,109.3 L169.1,108.5 L159.6,108.1 L150.0,107.9 L140.4,108.1 L130.9,108.5 L121.5,109.3 L112.4,110.4 L103.5,111.8 L95.0,113.5 L86.9,115.5 L79.3,117.8 L72.2,120.2 L65.7,122.9 L59.9,125.9 L54.7,129.0 L50.3,132.2 L46.6,135.6 L43.7,139.1 L41.7,142.7 L40.4,146.3 L40.0,150.0 L40.4,153.7 L41.7,157.3 L43.7,160.9 L46.6,164.4 L50.3,167.8 L54.7,171.0 L59.9,174.1 L65.7,177.1 L72.2,179.8 L79.3,182.2 L86.9,184.5 L95.0,186.5 L103.5,188.2 L112.4,189.6 L121.5,190.7 L130.9,191.5 L140.4,191.9 L150.0,192.1 L159.6,191.9 L169.1,191.5 L178.5,190.7 L187.6,189.6 L196.5,188.2 L205.0,186.5 L213.1,184.5 L220.7,182.2 L227.8,179.8 L234.3,177.1 L240.1,174.1 L245.3,171.0 Z" fill="none" stroke="#AAAAAA" stroke-width="0.5" opacity="0.4"/>
<path d="M245.3,171.0 L244.9,162.1 L243.8,153.1 L242.0,144.0 L239.5,135.0 L236.3,126.1 L232.5,117.4 L228.0,109.0 L223.0,100.8 L217.4,93.0 L211.2,85.7 L204.6,78.8 L197.6,72.5 L190.3,66.8 L182.6,61.7 L174.7,57.3 L166.5,53.6 L158.3,50.6 L150.0,48.4 L141.7,46.9 L133.5,46.3 L125.3,46.4 L117.4,47.3 L109.7,49.0 L102.4,51.5 L95.4,54.7 L88.8,58.6 L82.6,63.3 L77.0,68.6 L72.0,74.5 L67.5,81.0 L63.7,88.0 L60.5,95.5 L58.0,103.4 L56.2,111.6 L55.1,120.2 L54.7,129.0 L55.1,137.9 L56.2,146.9 L58.0,156.0 L60.5,165.0 L63.7,173.9 L67.5,182.6 L72.0,191.0 L77.0,199.2 L82.6,207.0 L88.8,214.3 L95.4,221.2 L102.4,227.5 L109.7,233.2 L117.4,238.3 L125.3,242.7 L133.5,246.4 L141.7,249.4 L150.0,251.6 L158.3,253.1 L166.5,253.7 L174.7,253.6 L182.6,252.7 L190.3,251.0 L197.6,248.5 L204.6,245.3 L211.2,241.4 L217.4,236.7 L223.0,231.4 L228.0,225.5 L232.5,219.0 L236.3,212.0 L239.5,204.5 L242.0,196.6 L243.8,188.4 L244.9,179.8 L245.3,171.0 Z" fill="none" stroke="#AAAAAA" stroke-width="0.5" opacity="0.4"/>
<path d="M205.0,113.5 L204.8,104.8 L204.2,96.5 L203.1,88.5 L201.7,81.0 L199.8,74.0 L197.6,67.6 L195.1,61.8 L192.1,56.7 L188.9,52.4 L185.4,48.7 L181.5,45.8 L177.5,43.8 L173.2,42.5 L168.8,42.0 L164.2,42.4 L159.6,43.6 L154.8,45.6 L150.0,48.4 L145.2,51.9 L140.4,56.2 L135.8,61.3 L131.2,67.0 L126.8,73.3 L122.5,80.2 L118.5,87.7 L114.6,95.6 L111.1,103.9 L107.9,112.6 L104.9,121.6 L102.4,130.8 L100.2,140.1 L98.3,149.5 L96.9,158.9 L95.8,168.3 L95.2,177.5 L95.0,186.5 L95.2,195.2 L95.8,203.5 L96.9,211.5 L98.3,219.0 L100.2,226.0 L102.4,232.4 L104.9,238.2 L107.9,243.3 L111.1,247.6 L114.6,251.3 L118.5,254.2 L122.5,256.2 L126.8,257.5 L131.2,258.0 L135.8,257.6 L140.4,256.4 L145.2,254.4 L150.0,251.6 L154.8,248.1 L159.6,243.8 L164.2,238.7 L168.8,233.0 L173.2,226.7 L177.5,219.8 L181.5,212.3 L185.4,204.4 L188.9,196.1 L192.1,187.4 L195.1,178.4 L197.6,169.2 L199.8,159.9 L201.7,150.5 L203.1,141.1 L204.2,131.7 L204.8,122.5 L205.0,113.5 Z" fill="none" stroke="#AAAAAA" stroke-width="0.5" opacity="0.4"/>
 
 
 
 
 |+⟩ 
 |-⟩ 
 |+i⟩ 
 |-i⟩ 
 |0⟩ 
 |1⟩ 
 
 
 
 
 
 H|0> 
<path d="M245.3,171.0 L249.7,167.8 L253.4,164.4 L256.3,160.9 L258.3,157.3 L259.6,153.7 L260.0,150.0 L259.6,146.3 L258.3,142.7 L256.3,139.1 L253.4,135.6 L249.7,132.2 L245.3,129.0 L240.1,125.9 L234.3,122.9 L227.8,120.2 L220.7,117.8 L213.1,115.5 L205.0,113.5 L196.5,111.8 L187.6,110.4 L178.5,109.3 L169.1,108.5 L159.6,108.1 L150.0,107.9 L140.4,108.1 L130.9,108.5 L121.5,109.3 L112.4,110.4 L103.5,111.8 L95.0,113.5 L86.9,115.5 L79.3,117.8 L72.2,120.2 L65.7,122.9 L59.9,125.9 L54.7,129.0 L50.3,132.2 L46.6,135.6 L43.7,139.1 L41.7,142.7 L40.4,146.3 L40.0,150.0 L40.4,153.7 L41.7,157.3 L43.7,160.9 L46.6,164.4 L50.3,167.8 L54.7,171.0 L59.9,174.1 L65.7,177.1 L72.2,179.8 L79.3,182.2 L86.9,184.5 L95.0,186.5 L103.5,188.2 L112.4,189.6 L121.5,190.7 L130.9,191.5 L140.4,191.9 L150.0,192.1 L159.6,191.9 L169.1,191.5 L178.5,190.7 L187.6,189.6 L196.5,188.2 L205.0,186.5 L213.1,184.5 L220.7,182.2 L227.8,179.8 L234.3,177.1 L240.1,174.1 L245.3,171.0 Z" fill="none" stroke=

## Power of Gates

Goqu provides `gate.Pow(g, k)` to raise a gate to an integer power by repeated matrix multiplication. This lets us verify algebraic identities:

- T^2 = S
- S^2 = Z
- T^4 = Z
- X^2 = I (self-inverse)
- H^2 = I (self-inverse)

In [8]:
%%
// Helper to compare matrices
matEq := func(a, b []complex128) bool {
	for i := range a {
		if cmplx.Abs(a[i]-b[i]) > 1e-10 {
			return false
		}
	}
	return true
}

// gate.Pow(T, 2) == S
fmt.Println("gate.Pow(T, 2) == S?", matEq(gate.Pow(gate.T, 2).Matrix(), gate.S.Matrix()))

// gate.Pow(S, 2) == Z
fmt.Println("gate.Pow(S, 2) == Z?", matEq(gate.Pow(gate.S, 2).Matrix(), gate.Z.Matrix()))

// gate.Pow(T, 4) == Z
fmt.Println("gate.Pow(T, 4) == Z?", matEq(gate.Pow(gate.T, 4).Matrix(), gate.Z.Matrix()))

// gate.Pow(X, 2) == I
fmt.Println("gate.Pow(X, 2) == I?", matEq(gate.Pow(gate.X, 2).Matrix(), gate.I.Matrix()))

// gate.Pow(H, 2) == I
fmt.Println("gate.Pow(H, 2) == I?", matEq(gate.Pow(gate.H, 2).Matrix(), gate.I.Matrix()))

gate.Pow(T, 2) == S? true
gate.Pow(S, 2) == Z? true
gate.Pow(T, 4) == Z? true
gate.Pow(X, 2) == I? true
gate.Pow(H, 2) == I? true


## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. What single gate is equivalent to applying T four times?
2. Can every single-qubit gate be decomposed into RY and RZ rotations? Why?
3. On the Bloch sphere, what axis and angle does the Z gate rotate around?

%%
// Exercise 1: Build H from RY and RZ
// Decomposition (up to global phase): H = RZ(pi) * RY(pi/2)
// In circuit order: first RY, then RZ
// Expected: P(|0>) = 0.5 and P(|1>) = 0.5, matching H|0>
//
// TODO: Fill in the rotation angles to reproduce H|0>
// cDecomp, _ := builder.New("h-decomp", 1).
// 	RY( /* angle */ , 0).
// 	RZ( /* angle */ , 0).
// 	Build()
//
// cH, _ := builder.New("h-direct", 1).H(0).Build()
//
// simD := statevector.New(1)
// simD.Evolve(cDecomp)
// svD := simD.StateVector()
//
// simH := statevector.New(1)
// simH.Evolve(cH)
// svH := simH.StateVector()
//
// fmt.Println("RZ*RY|0> =", svD)
// fmt.Println("H|0>     =", svH)
//
// // Check measurement probabilities match
// p0D := real(svD[0])*real(svD[0]) + imag(svD[0])*imag(svD[0])
// p1D := real(svD[1])*real(svD[1]) + imag(svD[1])*imag(svD[1])
// p0H := real(svH[0])*real(svH[0]) + imag(svH[0])*imag(svH[0])
// p1H := real(svH[1])*real(svH[1]) + imag(svH[1])*imag(svH[1])
// fmt.Printf("\nDecomposed: P(|0>)=%.4f, P(|1>)=%.4f\n", p0D, p1D)
// fmt.Printf("Hadamard:   P(|0>)=%.4f, P(|1>)=%.4f\n", p0H, p1H)
// fmt.Println("Probabilities match:", math.Abs(p0D-p0H) < 1e-10 && math.Abs(p1D-p1H) < 1e-10)
// fmt.Println("\nDecomposed circuit:")
// fmt.Println(draw.String(cDecomp))
fmt.Println("Uncomment the code above and fill in the rotation angles!")

In [9]:
%%
// Exercise 1: Build H from RY and RZ
// Decomposition (up to global phase): H = RZ(pi) * RY(pi/2)
// In circuit order: first RY, then RZ
//
// TODO: Fill in the rotation angles to reproduce H|0>
// cDecomp, _ := builder.New("h-decomp", 1).
// 	RY( /* angle */ , 0).
// 	RZ( /* angle */ , 0).
// 	Build()
//
// cH, _ := builder.New("h-direct", 1).H(0).Build()
//
// simD := statevector.New(1)
// simD.Evolve(cDecomp)
// svD := simD.StateVector()
//
// simH := statevector.New(1)
// simH.Evolve(cH)
// svH := simH.StateVector()
//
// fmt.Println("RZ*RY|0> =", svD)
// fmt.Println("H|0>     =", svH)
//
// // Check measurement probabilities match
// p0D := real(svD[0])*real(svD[0]) + imag(svD[0])*imag(svD[0])
// p1D := real(svD[1])*real(svD[1]) + imag(svD[1])*imag(svD[1])
// p0H := real(svH[0])*real(svH[0]) + imag(svH[0])*imag(svH[0])
// p1H := real(svH[1])*real(svH[1]) + imag(svH[1])*imag(svH[1])
// fmt.Printf("\nDecomposed: P(|0>)=%.4f, P(|1>)=%.4f\n", p0D, p1D)
// fmt.Printf("Hadamard:   P(|0>)=%.4f, P(|1>)=%.4f\n", p0H, p1H)
// fmt.Println("Probabilities match:", math.Abs(p0D-p0H) < 1e-10 && math.Abs(p1D-p1H) < 1e-10)
// fmt.Println("\nDecomposed circuit:")
// fmt.Println(draw.String(cDecomp))
fmt.Println("Uncomment the code above and fill in the rotation angles!")

Uncomment the code above and fill in the rotation angles!


In [10]:
%%
// Exercise 2: Use U3 to produce various states on the Bloch sphere
// U3(theta, phi, lambda) is the most general single-qubit gate.
//
// TODO: Find the U3 parameters that produce each target state:
//   State 1: |1>   (south pole)       -- Expected: P(|0>)=0, P(|1>)=1
//   State 2: |+>   (equator, +X axis) -- Expected: P(|0>)=0.5, P(|1>)=0.5
//   State 3: |-i>  = (|0> - i|1>)/sqrt(2)  (equator, -Y axis) -- Expected: P(|0>)=0.5, P(|1>)=0.5
//   State 4: An arbitrary point of your choice
//
// Hint: U3(theta, phi, lambda)|0> = cos(theta/2)|0> + e^(i*phi)*sin(theta/2)|1>
//
// c1, _ := builder.New("u3-ket1", 1).U3( /* theta, phi, lambda */ , 0).Build()
// sim1 := statevector.New(1)
// sim1.Evolve(c1)
// sv1 := sim1.StateVector()
// fmt.Println("U3(...)|0> =", sv1, "  (should be |1>)")
//
// c2, _ := builder.New("u3-plus", 1).U3( /* theta, phi, lambda */ , 0).Build()
// sim2 := statevector.New(1)
// sim2.Evolve(c2)
// sv2 := sim2.StateVector()
// fmt.Println("U3(...)|0> =", sv2, "  (should be |+>)")
//
// c3, _ := builder.New("u3-mi", 1).U3( /* theta, phi, lambda */ , 0).Build()
// sim3 := statevector.New(1)
// sim3.Evolve(c3)
// sv3 := sim3.StateVector()
// fmt.Println("U3(...)|0> =", sv3, "  (should be |-i>)")
//
// c4, _ := builder.New("u3-arb", 1).U3( /* theta, phi, lambda */ , 0).Build()
// sim4 := statevector.New(1)
// sim4.Evolve(c4)
// sv4 := sim4.StateVector()
// fmt.Printf("U3(...)|0> = [%.4f, %.4f]\n", sv4[0], sv4[1])
//
// gonbui.DisplayHTML(
// 	"<div style='display:flex;flex-wrap:wrap;gap:10px'>" +
// 		viz.Bloch(sv1, viz.WithTitle("|1>"), viz.WithSize(280, 280)) +
// 		viz.Bloch(sv2, viz.WithTitle("|+>"), viz.WithSize(280, 280)) +
// 		viz.Bloch(sv3, viz.WithTitle("|-i>"), viz.WithSize(280, 280)) +
// 		viz.Bloch(sv4, viz.WithTitle("Arbitrary"), viz.WithSize(280, 280)) +
// 		"</div>")
fmt.Println("Uncomment the code above and fill in the U3 parameters!")

Uncomment the code above and fill in the U3 parameters!


In [11]:
%%
// Exercise 2: Use U3 to produce various states on the Bloch sphere
// U3(theta, phi, lambda) is the most general single-qubit gate.
//
// TODO: Find the U3 parameters that produce each target state:
//   State 1: |1>   (south pole)
//   State 2: |+>   (equator, +X axis)
//   State 3: |-i>  = (|0> - i|1>)/sqrt(2)  (equator, -Y axis)
//   State 4: An arbitrary point of your choice
//
// Hint: U3(theta, phi, lambda)|0> = cos(theta/2)|0> + e^(i*phi)*sin(theta/2)|1>
//
// c1, _ := builder.New("u3-ket1", 1).U3( /* theta, phi, lambda */ , 0).Build()
// sim1 := statevector.New(1)
// sim1.Evolve(c1)
// sv1 := sim1.StateVector()
// fmt.Println("U3(...)|0> =", sv1, "  (should be |1>)")
//
// c2, _ := builder.New("u3-plus", 1).U3( /* theta, phi, lambda */ , 0).Build()
// sim2 := statevector.New(1)
// sim2.Evolve(c2)
// sv2 := sim2.StateVector()
// fmt.Println("U3(...)|0> =", sv2, "  (should be |+>)")
//
// c3, _ := builder.New("u3-mi", 1).U3( /* theta, phi, lambda */ , 0).Build()
// sim3 := statevector.New(1)
// sim3.Evolve(c3)
// sv3 := sim3.StateVector()
// fmt.Println("U3(...)|0> =", sv3, "  (should be |-i>)")
//
// c4, _ := builder.New("u3-arb", 1).U3( /* theta, phi, lambda */ , 0).Build()
// sim4 := statevector.New(1)
// sim4.Evolve(c4)
// sv4 := sim4.StateVector()
// fmt.Printf("U3(...)|0> = [%.4f, %.4f]\n", sv4[0], sv4[1])
//
// gonbui.DisplayHTML(
// 	"<div style='display:flex;flex-wrap:wrap;gap:10px'>" +
// 		viz.Bloch(sv1, viz.WithTitle("|1>"), viz.WithSize(280, 280)) +
// 		viz.Bloch(sv2, viz.WithTitle("|+>"), viz.WithSize(280, 280)) +
// 		viz.Bloch(sv3, viz.WithTitle("|-i>"), viz.WithSize(280, 280)) +
// 		viz.Bloch(sv4, viz.WithTitle("Arbitrary"), viz.WithSize(280, 280)) +
// 		"</div>")
fmt.Println("Uncomment the code above and fill in the U3 parameters!")

Uncomment the code above and fill in the U3 parameters!


## Key Takeaways

1. **Every single-qubit gate is a rotation** of the Bloch sphere, fully described by a 2x2 unitary matrix.

2. **Pauli gates** (X, Y, Z) are 180-degree rotations and are self-inverse.

3. **Hadamard** creates equal superposition and is the gateway to quantum parallelism.

4. **Phase gates** form a hierarchy: T^2 = S, S^2 = Z. The T gate is essential for universality.

5. **Rotation gates** (RX, RY, RZ) provide continuous rotations and are universal: any single-qubit unitary = RZ * RY * RZ (Euler decomposition).

6. **U3(theta, phi, lambda)** is the most general single-qubit gate -- it can reach any point on the Bloch sphere from |0>.

7. **Inverses** undo gates (gate followed by its inverse = identity). Use `g.Inverse()` in Goqu.

8. **Powers** let you compose a gate with itself: `gate.Pow(g, k)` multiplies the matrix k times.

In the next notebook, we will combine single-qubit gates with multi-qubit gates to create **entanglement** -- correlations that have no classical analog.